# Crisis Detector & Adaptive Threshold

## Purpose
This module is a **real-time crisis detection system** that adjusts the fake news classifier's decision threshold based on current events.

The core insight: **fake news spreads faster and in higher volume during crises** (wars, elections, pandemics). By detecting when a crisis is occurring, we can automatically tighten the classification threshold — accepting more false positives to catch more misinformation — rather than using a fixed 0.5 cutoff at all times.

## How it fits into the project
```
RSS News Feeds
      │
      ▼
CrisisDetector.fetch_news_stream()
      │
      ▼
CrisisDetector.calculate_crisis_score()   ← this notebook
      │
      ├── crisis_score:         0.0 – 1.0
      ├── crisis_type:          war_conflict / pandemic / election / ...
      └── adaptive_threshold:   0.3 (crisis) / 0.4 (elevated) / 0.5 (normal)
                │
                ▼
        Fake News Classifier
        (threshold adjusted dynamically)
```
Riddhi please check this !!

## How Crisis Detection Works

Crisis detection is based on a **weighted combination of 4 real-time signals**, each measuring a different aspect of the news environment.

### The 4 Signals

| # | Signal | What it measures | Method |
|---|---|---|---|
| 1 | `volume_spike` | Unusual surge in article count vs. baseline | Hourly bucket comparison |
| 2 | `sentiment_shift` | Negative tone in headlines | TextBlob polarity |
| 3 | `keyword_clustering` | Crisis-related content in headlines | Zero-shot NLI classifier (primary) / keyword fallback |
| 4 | `source_diversity` | Number of independent outlets reporting | Unique source count |

### Weighted Score Formula

```
crisis_score = (volume_spike × 0.25)
             + (sentiment_shift × 0.15)
             + (keyword_clustering × 0.45)   ← highest weight: most reliable signal
             + (source_diversity × 0.15)
```

A **consensus boost** of up to +0.20 is applied on top if >25% of headlines agree on the same crisis category at >60% model confidence.

### Adaptive Threshold Logic

| Crisis Score | Mode | Threshold | Effect |
|---|---|---|---|
| > 0.7 | 🔴 High crisis | 0.3 | Very strict — catches more fake news |
| 0.4 – 0.7 | 🟡 Elevated | 0.4 | Moderately strict |
| < 0.4 | 🟢 Normal | 0.5 | Standard classification |

### Signal Weight Rationale
- **`keyword_clustering` (0.45)** — the NLI model directly reads headline content with full context. Most reliable signal.
- **`volume_spike` (0.25)** — good for detecting *new* breaking events but unreliable for *sustained* crises, because the baseline naturally rises.
- **`source_diversity` (0.15)** — when multiple independent outlets cover the same event, that independently confirms the event is real.
- **`sentiment_shift` (0.15)** — news language is deliberately neutral even during crises ("Iran reports missile strike on..."), so TextBlob sentiment is a weak signal here. Retained as a minor boost.

## Known Limitations & Open Questions

### 1. Crisis keyword detection (resolved — see NLI section below)
The original approach used hardcoded substring matching, which caused false positives like:
> *"Meta **strikes** $60bn chip deal with AMD"* → wrongly flagged as `war_conflict`

This has been replaced with a zero-shot NLI classifier. Keyword lists are kept only as a fallback.

---

### 2. Signal weights — justification needed

**Open questions for Marcus and Ramiro:**

The current weights (`keyword_clustering=0.45`, `volume_spike=0.25`, etc.) are based on reasoning about each signal's reliability. However, they have **not been empirically validated** against a labelled dataset. Could you maybe find a way to make this better?

Can you find academic literature or prior work on crisis-detection weighting schemes (e.g., from GDELT, ACLED, or media monitoring research) that could give us a more principled basis for these values? Ideally something we can cite in the report.

---

### 3. Time window — hourly vs. daily buckets
`detect_unusual_volume()` currently uses **hourly buckets** to detect article volume spikes.

- **Hourly** → good for fast-breaking events (terrorist attack, election result)
- **Daily** → better for slow-moving crises (pandemic wave, economic deterioration)

Consider making this configurable, or running both and taking the max.

---

### 4. News sources — limited coverage
Currently pulling from 5 RSS feeds: Reuters World, Reuters Top, BBC World, Guardian World, AP News.

Adding more sources would improve the `source_diversity` signal. Candidates: Al Jazeera, NPR, NYT, DW, France24.

---

### 5. Sustained vs. new crises
`volume_spike` is blind to sustained crises — once a war has been running for weeks, the news volume baseline has already risen, so there's no spike to detect. The NLI classifier and consensus boost partially compensate for this, but it remains a fundamental limitation of the volume signal.

---
## Setup & Dependencies

Install the required packages before running this notebook.

```bash
pip install feedparser textblob torch transformers numpy pandas requests
```

| Package | Purpose |
|---|---|
| `feedparser` | Parse RSS feeds from news outlets |
| `textblob` | Sentiment analysis for Signal 2 |
| `torch` | Required backend for the NLI model |
| `transformers` | Hugging Face library — loads `facebook/bart-large-mnli` |
| `numpy` / `pandas` | Numerical operations |
| `requests` | HTTP requests for feeds |

> **Note:** The first time `CrisisDetector()` is instantiated, it downloads `facebook/bart-large-mnli` (~1.6 GB). Subsequent runs use the local cache and load in ~10–15 seconds.

## Zero-Shot NLI Classifier — Signal 3 in Detail

### Why not just keywords?
Substring matching has no understanding of context:

| Headline | Keyword matched | Correct? |
|---|---|---|
| *"Meta **strikes** $60bn chip deal with AMD"* | `strike` → `war_conflict` | ❌ False positive |
| *"Ukraine reports missile **strike** on Kyiv"* | `strike` → `war_conflict` | ✅ Correct |

Both sentences contain the same word. The system can't tell them apart without reading the sentence.

---

### What the NLI model does instead
We use [`facebook/bart-large-mnli`](https://huggingface.co/facebook/bart-large-mnli) — a Natural Language Inference model trained on over 400k sentence pairs.

It reads each headline and independently scores how strongly it *entails* each of our crisis labels:

```
Labels used:
  - "election fraud or voting irregularity"
  - "pandemic disease outbreak or public health emergency"
  - "war military conflict or armed attack"
  - "natural disaster earthquake flood hurricane"
  - "economic recession financial crash or market crisis"
  - "normal everyday news"   ← non-crisis catch-all
```

We use `multi_label=True`, meaning each label is scored **independently** (0–1), not in competition:

```
Headline: "Iran reports missile strike on Kyiv"
  war military conflict or armed attack  →  0.91  ✅ cleared threshold → war_conflict
  normal everyday news                   →  0.23
  economic recession...                  →  0.08
```

```
Headline: "Meta strikes $60bn chip deal with AMD"
  war military conflict or armed attack  →  0.18  ✗ below threshold → ignored
  normal everyday news                   →  0.88
  economic recession...                  →  0.41
```

A headline is only counted as a crisis hit if its top crisis label scores **≥ 0.45**.

---

### Crisis content score formula

```
keyword_clustering = (crisis_hits / total_articles) × avg_confidence_of_crisis_hits
```

This separates *coverage* (what fraction of news is crisis-related) from *confidence* (how sure the model is). Both need to be high to produce a high score.

---

### Consensus boost
During a sustained crisis, only a fraction of all world news covers it — so raw coverage is moderate (~30%) even when the model is highly confident. The consensus boost adds up to +0.20 if:
- **>25%** of headlines classify to the same crisis category, **AND**
- their **average confidence is >0.60**

---

### Fallback behaviour
If `torch` or `transformers` are not installed, the system automatically falls back to keyword matching with a warning message. No code changes needed.

**Dependencies:** `pip install torch transformers`  
**Model size:** ~1.6 GB (downloaded and cached on first run)

In [7]:
# crisis_detector
import requests
import feedparser
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
from collections import defaultdict
import time
from textblob import TextBlob  # For sentiment analysis
from transformers import pipeline  # For zero-shot classification

---
## Implementation

The `CrisisDetector` class is split across **5 code cells** below. Run them all top-to-bottom before using the class.

| Cell | Section | Methods |
|---|---|---|
| **Class definition** | `__init__` | Initialises labels, keyword fallback lists, RSS feed config, and internal state |
| **Section 1** | Classifier loading & zero-shot inference | `_load_zero_shot_classifier()`, `_classify_headlines_zero_shot()` |
| **Section 2** | News fetching & signals 1–4 | `fetch_news_stream()`, `detect_unusual_volume()`, `track_sentiment_over_time()`, `detect_crisis_keywords()`, `_detect_crisis_zero_shot_score()`, `_detect_crisis_keywords_fallback()`, `calculate_source_diversity()` |
| **Section 3** | Main scoring | `calculate_crisis_score()`, `_apply_consensus_boost()` |
| **Section 4** | Crisis type detection & output helpers | `determine_crisis_type_with_keywords()`, `_determine_crisis_type_zero_shot()`, `_determine_crisis_type_keyword_fallback()`, `generate_explanations()`, `get_sample_headlines()`, `monitor_continuously()` |

Each section ends with lines like `CrisisDetector.method = method` that attach the functions to the class. This is a standard notebook pattern for splitting a class across cells without losing readability.


In [8]:
class CrisisDetector:
    """
    Real-time crisis detection system.

    Combines 4 signals (volume spike, sentiment shift, keyword clustering,
    source diversity) into a single crisis score that adjusts the fake news
    classifier's decision threshold.

    The class is split across several cells below for readability. Run all
    cells top-to-bottom before using the class.
    """

    def __init__(self):
        # Zero-shot NLI labels and their short category names
        self.crisis_labels = [
            'election fraud or voting irregularity',
            'pandemic disease outbreak or public health emergency',
            'war military conflict or armed attack',
            'natural disaster earthquake flood hurricane',
            'economic recession financial crash or market crisis',
            'normal everyday news'
        ]
        self.label_to_category = {
            'election fraud or voting irregularity': 'election',
            'pandemic disease outbreak or public health emergency': 'pandemic',
            'war military conflict or armed attack': 'war_conflict',
            'natural disaster earthquake flood hurricane': 'natural_disaster',
            'economic recession financial crash or market crisis': 'economic',
            'normal everyday news': 'none'
        }

        # Keyword lists — used as fallback if the NLI model is unavailable
        self.crisis_keywords = {
            'election': [
                'election', 'vote', 'ballot', 'campaign', 'polls',
                'presidential', 'democracy', 'voting', 'fraud', 'results'
            ],
            'pandemic': [
                'pandemic', 'epidemic', 'outbreak', 'virus', 'covid',
                'ebola', 'quarantine', 'lockdown', 'vaccine', 'cases',
                'infection', 'hospital', 'icu', 'death toll'
            ],
            'war_conflict': [
                'war', 'airstrike', 'bombing', 'invasion',
                'military offensive', 'troops deployed', 'rebel forces',
                'missile strike', 'explosion kills', 'armed conflict',
                'combat casualties', 'refugees flee'
            ],
            'natural_disaster': [
                'earthquake', 'hurricane', 'flood', 'tsunami', 'wildfire',
                'tornado', 'landslide', 'volcano', 'drought', 'storm surge'
            ],
            'economic': [
                'recession', 'market crash', 'financial crisis', 'inflation surge',
                'bankruptcy', 'stock market', 'economic collapse', 'debt default',
                'unemployment rate', 'bank failure'
            ]
        }

        # RSS feeds — the raw data source
        self.rss_feeds = {
            'reuters_world': 'http://feeds.reuters.com/Reuters/worldNews',
            'reuters_top':   'http://feeds.reuters.com/reuters/topNews',
            'bbc_world':     'http://feeds.bbci.co.uk/news/world/rss.xml',
            'guardian_world': 'https://www.theguardian.com/world/rss',
            'ap_news':       'https://feeds.feedburner.com/AssociatedPressTopNews'
        }

        # Internal state
        self.news_history    = []
        self.crisis_mode     = False
        self.current_crisis_type = 'none'
        self.crisis_score    = 0.0

        # Load the NLI model (defined in the next cell)
        self.zero_shot_classifier = None
        self._load_zero_shot_classifier()


In [9]:
# ─────────────────────────────────────────────────────────────
# SECTION 1 — Classifier Loading & Zero-Shot Inference
# ─────────────────────────────────────────────────────────────

def _load_zero_shot_classifier(self):
    """
    Load facebook/bart-large-mnli for zero-shot classification.
    On first run this downloads ~1.6 GB; subsequent runs use the local cache.
    Falls back to keyword matching if torch / transformers are not installed.
    """
    try:
        import torch  # noqa: F401
        from transformers import pipeline as hf_pipeline
        print("Loading zero-shot classifier (first run downloads ~1.6 GB)...")
        self.zero_shot_classifier = hf_pipeline(
            "zero-shot-classification",
            model="facebook/bart-large-mnli",
            device=-1  # CPU; set to 0 for GPU
        )
        print("✅ Zero-shot classifier loaded — context-aware crisis detection active")
    except ImportError as e:
        missing = "torch" if "torch" in str(e) else "transformers"
        print(f"⚠️  Missing dependency: {e}")
        print(f"   Fix: pip install {missing}")
        print("   Falling back to keyword matching.")
        self.zero_shot_classifier = None
    except Exception as e:
        print(f"⚠️  Could not load zero-shot classifier: {e}")
        print("   Falling back to keyword matching.")
        self.zero_shot_classifier = None


def _classify_headlines_zero_shot(self, news_stream, max_articles=40):
    """
    Run zero-shot classification on each headline using multi_label=True.

    WHY multi_label=True:
      Standard (multi_label=False) forces labels to compete — "war conflict"
      vs "normal everyday news". A genuine war headline can still lose to
      "normal" if it's phrased neutrally ("Iran peace talks stall").

      multi_label=True scores each label independently (0–1). A headline
      counts as a crisis hit if its top crisis label scores ≥ CRISIS_THRESHOLD.
      This is far more sensitive during sustained crises where world news is
      mixed with other topics.
    """
    CRISIS_THRESHOLD = 0.45  # Minimum independent score to count as a crisis hit

    results = []
    for article in news_stream[:max_articles]:
        try:
            out = self.zero_shot_classifier(
                article['title'],
                candidate_labels=self.crisis_labels,
                multi_label=True
            )
            scores = dict(zip(out['labels'], out['scores']))

            # Highest-scoring crisis label (excluding the normal-news catch-all)
            crisis_scores = {lbl: scr for lbl, scr in scores.items()
                             if lbl != 'normal everyday news'}
            top_crisis_label = max(crisis_scores, key=crisis_scores.get)
            top_crisis_score = crisis_scores[top_crisis_label]

            predicted_category = (
                self.label_to_category.get(top_crisis_label, 'none')
                if top_crisis_score >= CRISIS_THRESHOLD
                else 'none'
            )

            results.append({
                'title': article['title'],
                'source': article['source'],
                'predicted_category': predicted_category,
                'top_label': top_crisis_label,
                'confidence': top_crisis_score,
                'all_scores': scores
            })
        except Exception:
            continue
    return results


# Attach methods to the class
CrisisDetector._load_zero_shot_classifier   = _load_zero_shot_classifier
CrisisDetector._classify_headlines_zero_shot = _classify_headlines_zero_shot


In [10]:
# ─────────────────────────────────────────────────────────────
# SECTION 2 — News Fetching & Signal Calculations (Signals 1–4)
# ─────────────────────────────────────────────────────────────

def fetch_news_stream(self, hours_back=6):
    """Fetch recent articles from all configured RSS feeds."""
    articles = []
    cutoff_time = datetime.now() - timedelta(hours=hours_back)

    for source_name, feed_url in self.rss_feeds.items():
        try:
            feed = feedparser.parse(feed_url)
            for entry in feed.entries[:20]:
                pub_time = (
                    datetime(*entry.published_parsed[:6])
                    if hasattr(entry, 'published_parsed')
                    else datetime.now()
                )
                if pub_time < cutoff_time:
                    continue
                articles.append({
                    'source':    source_name,
                    'title':     entry.title,
                    'summary':   entry.get('summary', ''),
                    'published': pub_time,
                    'content':   entry.title + " " + entry.get('summary', '')
                })
        except Exception as e:
            print(f"Error fetching {source_name}: {e}")

    return articles


# ── Signal 1: Volume spike ────────────────────────────────────

def detect_unusual_volume(self, news_stream):
    """Compare current-hour article count against recent hourly baseline."""
    if not news_stream:
        return 0.0

    now = datetime.now()
    articles_by_hour = defaultdict(int)
    for article in news_stream:
        articles_by_hour[article['published'].strftime('%Y-%m-%d %H:00')] += 1

    current_hour   = now.strftime('%Y-%m-%d %H:00')
    current_volume = articles_by_hour.get(current_hour, 0)
    previous_hours = [h for h in articles_by_hour if h != current_hour]
    avg_volume     = np.mean([articles_by_hour[h] for h in previous_hours]) if previous_hours else 1

    spike_ratio = current_volume / avg_volume if avg_volume > 0 else 1
    return min(spike_ratio / 3, 1.0)


# ── Signal 2: Sentiment shift ─────────────────────────────────

def track_sentiment_over_time(self, news_stream):
    """
    TextBlob average polarity across headlines.
    NOTE: News is written in neutral journalistic tone even during crises,
    so this signal has low sensitivity. Weight kept at 0.15.
    """
    if not news_stream:
        return 0.0

    sentiments = []
    for article in news_stream[:50]:
        try:
            sentiments.append(TextBlob(article['title']).sentiment.polarity)
        except Exception:
            continue

    if not sentiments:
        return 0.0

    avg_sentiment = np.mean(sentiments)
    return max(0, min(1, 0.5 - (avg_sentiment * 0.5)))


# ── Signal 3: Keyword / NLI clustering ───────────────────────

def detect_crisis_keywords(self, news_stream, zs_results=None):
    """Dispatch to NLI scorer (primary) or keyword density (fallback)."""
    if zs_results is not None:
        return self._detect_crisis_zero_shot_score(zs_results)
    return self._detect_crisis_keywords_fallback(news_stream)


def _detect_crisis_zero_shot_score(self, zs_results):
    """
    Score = (crisis hits / total articles) × avg confidence of those hits.

    Separates *coverage* (fraction of news that is crisis-related) from
    *confidence* (how certain the model is). Both must be high to score high.
    """
    if not zs_results:
        return 0.0

    crisis_hits = [r for r in zs_results if r['predicted_category'] != 'none']
    if not crisis_hits:
        return 0.0

    coverage       = len(crisis_hits) / len(zs_results)
    avg_confidence = np.mean([r['confidence'] for r in crisis_hits])
    return min(coverage * avg_confidence, 1.0)


def _detect_crisis_keywords_fallback(self, news_stream):
    """Fallback: fraction of articles that match at least one keyword."""
    if not news_stream:
        return 0.0

    keyword_counts = defaultdict(int)
    for article in news_stream:
        content = article['title'].lower() + " " + article.get('summary', '').lower()
        for category, keywords in self.crisis_keywords.items():
            for keyword in keywords:
                if keyword in content:
                    keyword_counts[category] += 1
                    break

    return min(sum(keyword_counts.values()) / len(news_stream), 1.0)


# ── Signal 4: Source diversity ────────────────────────────────

def calculate_source_diversity(self, news_stream):
    """Fraction of the 5 configured outlets that reported at least one article."""
    if not news_stream:
        return 0.0
    return min(len(set(a['source'] for a in news_stream)) / 5, 1.0)


# Attach methods to the class
CrisisDetector.fetch_news_stream             = fetch_news_stream
CrisisDetector.detect_unusual_volume         = detect_unusual_volume
CrisisDetector.track_sentiment_over_time     = track_sentiment_over_time
CrisisDetector.detect_crisis_keywords        = detect_crisis_keywords
CrisisDetector._detect_crisis_zero_shot_score   = _detect_crisis_zero_shot_score
CrisisDetector._detect_crisis_keywords_fallback = _detect_crisis_keywords_fallback
CrisisDetector.calculate_source_diversity    = calculate_source_diversity


In [11]:
# ─────────────────────────────────────────────────────────────
# SECTION 3 — Main Scoring: calculate_crisis_score
# ─────────────────────────────────────────────────────────────

# Signal weights — rationale documented in the "How it Works" cell above
SIGNAL_WEIGHTS = {
    'volume_spike':       0.25,   # good for breaking events; blind to sustained crises
    'sentiment_shift':    0.15,   # weak on neutral journalistic language
    'keyword_clustering': 0.45,   # most reliable — NLI reads full context
    'source_diversity':   0.15,   # independent corroboration across outlets
}


def calculate_crisis_score(self, news_stream):
    """
    Aggregate all 4 signals into a single crisis score (0–1) and
    return the corresponding adaptive threshold.
    """
    if not news_stream:
        return {
            'crisis_score': 0.0, 'crisis_type': 'none', 'crisis_mode': False,
            'adaptive_threshold': 0.5, 'signals': {}, 'signal_weights': {},
            'keyword_details': {'top_keywords': [], 'all_counts': {}},
            'sample_headlines': [], 'explanations': ['No news data available'],
            'article_count': 0, 'classifier_used': 'none'
        }

    # Run NLI inference (if model loaded)
    zs_results = None
    if self.zero_shot_classifier is not None:
        print("  Running zero-shot classification on headlines...")
        zs_results = self._classify_headlines_zero_shot(news_stream)

    # Compute the 4 signals
    signals = {
        'volume_spike':       self.detect_unusual_volume(news_stream),
        'sentiment_shift':    self.track_sentiment_over_time(news_stream),
        'keyword_clustering': self.detect_crisis_keywords(news_stream, zs_results),
        'source_diversity':   self.calculate_source_diversity(news_stream),
    }

    # Weighted sum
    crisis_score = sum(signals[s] * SIGNAL_WEIGHTS[s] for s in signals)

    # Consensus boost: reward a strong, consistent NLI signal
    if zs_results:
        crisis_score = self._apply_consensus_boost(crisis_score, zs_results)

    # Determine crisis type and build output
    crisis_type, keyword_details = self.determine_crisis_type_with_keywords(news_stream, zs_results)
    sample_headlines  = self.get_sample_headlines(news_stream, crisis_type, zs_results)
    classifier_used   = 'zero-shot NLI (multi-label)' if zs_results is not None else 'keyword fallback'
    explanations      = self.generate_explanations(signals, SIGNAL_WEIGHTS, crisis_score, crisis_type, classifier_used)

    # Map score → adaptive threshold
    if crisis_score > 0.7:
        adaptive_threshold, crisis_mode = 0.3, True
    elif crisis_score > 0.4:
        adaptive_threshold, crisis_mode = 0.4, True
    else:
        adaptive_threshold, crisis_mode = 0.5, False

    # Persist state
    self.crisis_score        = crisis_score
    self.current_crisis_type = crisis_type
    self.crisis_mode         = crisis_mode
    self.adaptive_threshold  = adaptive_threshold

    return {
        'crisis_score':      crisis_score,
        'crisis_type':       crisis_type,
        'crisis_mode':       crisis_mode,
        'adaptive_threshold': adaptive_threshold,
        'signals':           signals,
        'signal_weights':    SIGNAL_WEIGHTS,
        'keyword_details':   keyword_details,
        'sample_headlines':  sample_headlines,
        'explanations':      explanations,
        'article_count':     len(news_stream),
        'classifier_used':   classifier_used,
    }


def _apply_consensus_boost(self, crisis_score, zs_results):
    """
    Add up to +0.20 when a clear NLI consensus exists.

    Problem this solves: even during a real war, only ~30% of world news
    headlines are about that war — so raw coverage dilutes the score.
    If >25% of articles agree on one category at avg confidence >0.60,
    we lift the score proportionally to reward the clear signal.

    Parameters
    ----------
    CONSENSUS_THRESHOLD : fraction of articles that must be crisis-related
    CONFIDENCE_THRESHOLD: minimum average NLI confidence required
    MAX_BOOST           : ceiling on the upward adjustment
    """
    CONSENSUS_THRESHOLD  = 0.25
    CONFIDENCE_THRESHOLD = 0.60
    MAX_BOOST            = 0.20

    if not zs_results:
        return crisis_score

    # Bucket confidence scores by category
    category_hits = defaultdict(list)
    for r in zs_results:
        if r['predicted_category'] != 'none':
            category_hits[r['predicted_category']].append(r['confidence'])

    if not category_hits:
        return crisis_score

    # Evaluate the dominant category
    dominant_cat   = max(category_hits, key=lambda c: len(category_hits[c]))
    hits           = category_hits[dominant_cat]
    proportion     = len(hits) / len(zs_results)
    avg_confidence = np.mean(hits)

    if proportion >= CONSENSUS_THRESHOLD and avg_confidence >= CONFIDENCE_THRESHOLD:
        coverage_factor   = min(proportion / CONSENSUS_THRESHOLD, 2.0)
        confidence_factor = min((avg_confidence - CONFIDENCE_THRESHOLD) / 0.40, 1.0)
        boost             = coverage_factor * confidence_factor * (MAX_BOOST / 2)
        crisis_score      = min(crisis_score + boost, 1.0)

    return crisis_score


# Attach methods to the class
CrisisDetector.calculate_crisis_score  = calculate_crisis_score
CrisisDetector._apply_consensus_boost  = _apply_consensus_boost


In [12]:
# ─────────────────────────────────────────────────────────────
# SECTION 4 — Crisis Type Detection & Output Helpers
# ─────────────────────────────────────────────────────────────

def determine_crisis_type_with_keywords(self, news_stream, zs_results=None):
    """Entry point: delegate to NLI or keyword fallback."""
    if zs_results is not None:
        return self._determine_crisis_type_zero_shot(zs_results)
    return self._determine_crisis_type_keyword_fallback(news_stream)


def _determine_crisis_type_zero_shot(self, zs_results):
    """Confidence-weighted voting: the category with the highest total confidence wins."""
    category_confidence = defaultdict(float)
    category_examples   = defaultdict(list)

    for r in zs_results:
        cat = r['predicted_category']
        if cat != 'none':
            category_confidence[cat] += r['confidence']
            category_examples[cat].append({
                'keyword':    r['top_label'],
                'title':      r['title'],
                'confidence': r['confidence']
            })

    if not category_confidence:
        return 'none', {'top_keywords': [], 'all_counts': {}}

    crisis_type = max(category_confidence, key=category_confidence.get)
    examples    = sorted(category_examples[crisis_type],
                         key=lambda x: x['confidence'], reverse=True)[:5]

    return crisis_type, {
        'top_keywords':    [(e['title'][:60], round(e['confidence'], 3)) for e in examples],
        'total_mentions':  len(category_examples[crisis_type]),
        'all_counts':      {k: round(v, 3) for k, v in category_confidence.items()},
        'method':          'zero-shot NLI (multi-label)'
    }


def _determine_crisis_type_keyword_fallback(self, news_stream):
    """Fallback: pick the category with the most keyword matches."""
    category_counts = defaultdict(int)
    keyword_hits    = defaultdict(list)

    for article in news_stream:
        content = article['title'].lower() + " " + article.get('summary', '').lower()
        for category, keywords in self.crisis_keywords.items():
            for keyword in keywords:
                if keyword in content:
                    category_counts[category] += 1
                    keyword_hits[category].append({'keyword': keyword, 'title': article['title']})
                    break

    if not category_counts:
        return 'none', {'top_keywords': [], 'all_counts': {}}

    crisis_type  = max(category_counts, key=category_counts.get)
    keyword_freq = defaultdict(int)
    for hit in keyword_hits.get(crisis_type, []):
        keyword_freq[hit['keyword']] += 1

    return crisis_type, {
        'top_keywords':   sorted(keyword_freq.items(), key=lambda x: x[1], reverse=True)[:5],
        'total_mentions': category_counts[crisis_type],
        'all_counts':     dict(category_counts),
        'method':         'keyword fallback'
    }


# ── Output helpers ────────────────────────────────────────────

def generate_explanations(self, signals, weights, crisis_score, crisis_type, classifier_used='keyword fallback'):
    """Return a list of plain-English strings explaining each signal's contribution."""
    explanations = [f"🤖 Classifier: {classifier_used}"]

    if signals['volume_spike'] > 0.7:
        explanations.append(f"📈 HIGH VOLUME: {signals['volume_spike']*100:.0f}% spike in news activity")
    elif signals['volume_spike'] > 0.3:
        explanations.append(f"📈 Elevated news volume ({signals['volume_spike']:.2f})")
    else:
        explanations.append(f"📉 No volume spike ({signals['volume_spike']:.2f}) — may be a sustained crisis, not a new one")

    if signals['sentiment_shift'] > 0.7:
        explanations.append("😠 VERY NEGATIVE sentiment detected")
    elif signals['sentiment_shift'] > 0.5:
        explanations.append(f"😐 Negative sentiment shift ({signals['sentiment_shift']:.2f})")
    else:
        explanations.append(f"😐 Neutral sentiment ({signals['sentiment_shift']:.2f}) — news is often neutral even during crises")

    if signals['keyword_clustering'] > 0.5:
        explanations.append(f"🔍 Strong {crisis_type} crisis signal ({signals['keyword_clustering']:.2f})")
    elif signals['keyword_clustering'] > 0.2:
        explanations.append(f"🔍 Moderate {crisis_type} signal ({signals['keyword_clustering']:.2f})")

    if signals['source_diversity'] > 0.6:
        explanations.append(f"🌐 Multiple sources reporting ({signals['source_diversity']*100:.0f}% diversity)")

    explanations.append(f"📊 Final crisis score: {crisis_score:.2f} (weighted combination)")
    return explanations


def get_sample_headlines(self, news_stream, crisis_type, zs_results=None, n=5):
    """Return up to n example headlines that triggered crisis detection."""
    if zs_results is not None:
        relevant = [
            {
                'title':           r['title'],
                'keyword_matched': f"{r['top_label']} ({r['confidence']:.0%})",
                'source':          r['source']
            }
            for r in zs_results if r['predicted_category'] == crisis_type
        ]
        relevant.sort(key=lambda x: x['keyword_matched'], reverse=True)
        return relevant[:n]

    samples  = []
    keywords = self.crisis_keywords.get(crisis_type, [])
    for article in news_stream:
        for keyword in keywords:
            if keyword in article['title'].lower():
                samples.append({'title': article['title'], 'keyword_matched': keyword, 'source': article['source']})
                break
        if len(samples) >= n:
            break
    return samples


def monitor_continuously(self, interval_minutes=15):
    """Polling loop — runs until interrupted with Ctrl-C."""
    print("Starting crisis monitor...")
    while True:
        try:
            news_stream = self.fetch_news_stream(hours_back=6)
            result      = self.calculate_crisis_score(news_stream)

            print(f"\n{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print(f"🤖 Classifier:         {result['classifier_used']}")
            print(f"🚨 Crisis Mode:        {result['crisis_mode']}")
            print(f"📊 Crisis Score:       {result['crisis_score']:.3f}")
            print(f"📍 Crisis Type:        {result['crisis_type']}")
            print(f"⚖️  Adaptive Threshold: {result['adaptive_threshold']}")
            for signal, value in result['signals'].items():
                print(f"  • {signal}: {value:.3f}")

            self.news_history.append({'timestamp': datetime.now(), 'result': result})
            cutoff = datetime.now() - timedelta(hours=24)
            self.news_history = [h for h in self.news_history if h['timestamp'] > cutoff]
            time.sleep(interval_minutes * 60)

        except KeyboardInterrupt:
            print("\nMonitoring stopped.")
            break
        except Exception as e:
            print(f"Error: {e}")
            time.sleep(60)


# Attach methods to the class
CrisisDetector.determine_crisis_type_with_keywords    = determine_crisis_type_with_keywords
CrisisDetector._determine_crisis_type_zero_shot       = _determine_crisis_type_zero_shot
CrisisDetector._determine_crisis_type_keyword_fallback = _determine_crisis_type_keyword_fallback
CrisisDetector.generate_explanations   = generate_explanations
CrisisDetector.get_sample_headlines    = get_sample_headlines
CrisisDetector.monitor_continuously    = monitor_continuously


---
## Run the Detector

The cell below **instantiates the detector, fetches live news and prints a full crisis report**.

On first run it will download the `facebook/bart-large-mnli` model (~1.6 GB). Subsequent runs load from cache.

### What the output shows
- **Classifier** — confirms whether zero-shot NLI or keyword fallback is active
- **Crisis Type** — the dominant crisis category detected (e.g. `WAR_CONFLICT`)
- **Crisis Score** — 0.0–1.0 weighted composite score
- **Crisis Mode** — `True` if score > 0.4
- **Adaptive Threshold** — the value passed to the fake news classifier
- **Detection method** — `zero-shot NLI (multi-label)` or `keyword fallback`
- **Top classified headlines** — the 5 most confidently matched crisis headlines with confidence %
- **Category scores** — confidence mass per crisis category (useful for diagnosis)
- **Signal Breakdown** — individual bar chart for each of the 4 signals
- **Explanations** — plain-English interpretation of each signal's contribution
- **Sample Headlines** — example articles that triggered the detection

### To use the adaptive threshold in the fake news classifier
```python
detector = CrisisDetector()
news_stream = detector.fetch_news_stream(hours_back=6)
result = detector.calculate_crisis_score(news_stream)

threshold = result['adaptive_threshold']   # 0.3 / 0.4 / 0.5
# Pass 'threshold' to your classifier's predict_proba cutoff
```

In [13]:
if __name__ == "__main__":
    detector = CrisisDetector()
    
    # Fetch news
    news_stream = detector.fetch_news_stream(hours_back=6)
    print(f"📰 Fetched {len(news_stream)} articles")
    
    if news_stream:
        result = detector.calculate_crisis_score(news_stream)
        
        print("\n" + "="*60)
        print("CRISIS DETECTION RESULTS")
        print("="*60)
        
        # Core results
        print(f"\n🤖 Classifier:         {result['classifier_used']}")
        print(f"📍 Crisis Type:        {result['crisis_type'].upper()}")
        print(f"📊 Crisis Score:       {result['crisis_score']:.3f}")
        print(f"🚨 Crisis Mode:        {result['crisis_mode']}")
        print(f"⚖️  Adaptive Threshold: {result['adaptive_threshold']}")
        
        # Detection method used
        method = result['keyword_details'].get('method', 'unknown')
        print(f"\n🔎 Detection method: {method}")
        
        # Top detected headlines / keywords
        if result['keyword_details']['top_keywords']:
            label = "Top classified headlines" if 'zero-shot' in method else "Top keywords"
            print(f"\n🔍 {label}:")
            for item, score in result['keyword_details']['top_keywords']:
                print(f"  • [{score}] {item}")
        
        # Category confidence breakdown
        if result['keyword_details'].get('all_counts'):
            print(f"\n📂 Category scores:")
            for cat, score in sorted(result['keyword_details']['all_counts'].items(),
                                     key=lambda x: x[1], reverse=True):
                print(f"  • {cat:20} {score}")
        
        # Signal breakdown
        print(f"\n📈 Signal Breakdown:")
        for signal, value in result['signals'].items():
            weight = result['signal_weights'][signal]
            bar = "█" * int(value * 20)
            print(f"  {signal:20} [{bar:20}] {value:.3f} (weight: {weight:.2f})")
        
        # Explanations
        print(f"\n💡 Explanations:")
        for exp in result['explanations']:
            print(f"  • {exp}")
        
        # Sample headlines
        if result['sample_headlines']:
            print(f"\n📰 Sample Headlines:")
            for headline in result['sample_headlines']:
                print(f"  • {headline['title'][:80]}")
                print(f"    (matched: '{headline['keyword_matched']}' from {headline['source']})")
    else:
        print("❌ No articles fetched")

Loading zero-shot classifier (first run downloads ~1.6 GB)...


Loading weights: 100%|██████████| 515/515 [00:00<00:00, 588.54it/s, Materializing param=model.shared.weight]                                   


✅ Zero-shot classifier loaded — context-aware crisis detection active
📰 Fetched 8 articles
  Running zero-shot classification on headlines...

CRISIS DETECTION RESULTS

🤖 Classifier:         zero-shot NLI (multi-label)
📍 Crisis Type:        WAR_CONFLICT
📊 Crisis Score:       0.406
🚨 Crisis Mode:        True
⚖️  Adaptive Threshold: 0.4

🔎 Detection method: zero-shot NLI (multi-label)

🔍 Top classified headlines:
  • [0.997] Why did US and Israel attack Iran and how long could the war
  • [0.876] Deadly Texas bar shooting 'potentially act of terrorism', FB
  • [0.862] Oil and gas prices jump as conflict escalates

📂 Category scores:
  • war_conflict         2.735

📈 Signal Breakdown:
  volume_spike         [                    ] 0.000 (weight: 0.25)
  sentiment_shift      [██████████          ] 0.501 (weight: 0.15)
  keyword_clustering   [██████              ] 0.342 (weight: 0.45)
  source_diversity     [████████            ] 0.400 (weight: 0.15)

💡 Explanations:
  • 🤖 Classifier: zero-s